In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
#Load all models and preprocessors
model = load_model('model.h5')
scaler = pickle.load(open('scaler.pkl', 'rb'))
onehot_encoder = pickle.load(open('onehot_encoder.pkl', 'rb'))
label_encoder = pickle.load(open('label_encoder.pkl', 'rb'))

In [4]:
# Example input data for prediction

input_data ={
    'CreditScore': 650,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 35,
    'Tenure': 5,
    'Balance': 10000.0,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000.0
    
}


In [11]:
# to work with input data, we need to encode columns such as geography into codings 
# so that it can be interpretaed by the model - we use label encoder here..



geography_encoded = onehot_encoder.transform([[input_data['Geography']]])
geography_encoded_df = pd.DataFrame(geography_encoded, columns= onehot_encoder.get_feature_names_out(['Geography']))

/Users/adyant/Desktop/ANN_ClassificationProject/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
#geography input data is now encoded and stored in geography_encoded_df, which can be used for further processing or prediction with the model.
geography_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [ ]:
#convert to dataframe for further processing
input_df = pd.DataFrame([input_data])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,650,France,Male,35,5,10000.0,2,1,1,50000.0


In [13]:
#Encode Gender column using label encoder
input_df['Gender'] = label_encoder.transform(input_df['Gender'])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,650,France,1,35,5,10000.0,2,1,1,50000.0


In [15]:
# now combine the encoded geography columns with the rest of the input data
#replace the original Geography column with the encoded columns of input_data
input_df = pd.concat([input_df.drop('Geography', axis=1), geography_encoded_df], axis=1)
input_df


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,650,1,35,5,10000.0,2,1,1,50000.0,1.0,0.0,0.0


In [16]:
# SCALING the data
input_scaled_df = scaler.transform(input_df)

In [17]:
input_scaled_df

array([[-0.01709861,  0.91324755, -0.37056859, -0.00134472, -1.05836066,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [18]:
#Now predicting churn
prediction = model.predict(input_scaled_df)

1/1 [==============================] - 0s 148ms/step


In [ ]:
prediction # it is a probability value, we can set a threshold to classify it as churn or not churn. For example, if the prediction is greater than 0.5, we can classify it as churn (1), otherwise not churn (0).

array([[0.00827652]], dtype=float32)

In [23]:
prediction_probability = prediction[0][0]

In [25]:
prediction_probability 

0.00827652

In [26]:
if prediction_probability > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
